In [2]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### DETECCION DE ANOMALIAS

##### 6-1 Introduccion

##### 6-2 Cuando usar SQL

Cuando sí usar SQL: Cuando los datos estan ya en una base de datos y no se requiere un analisis estadistico profundo. 

Cuando no usar SQL: SQL no puede implementar analisis estadistico o de aprendizaje de maquina o deteccion de anomalias en tiempo real.

##### 6-3 Detectar Anomalias por Ordenamiento

En este capitulo se usaran los datos de registros de tormentas

In [4]:
# Visualizar la tabla de tormentas

query = r"""
select
    *
from
    anomalias.storms;
"""
df = pd.read_sql(text(query),engine)
df.head()

,storms_id,name,year,month,day,hour,lat,long,status,category,wind,pressure,tropicalstorm_force_diameter,hurricane_force_diameter,location
0,1,AL011852,1852,8,26,6,30.2,-88.6,hurricane,3,100,961,NA,NA,Central of USA
1,2,AL031853,1853,9,3,12,19.7,-56.2,hurricane,4,130,924,NA,NA,Near the east coast of Costa Rica
2,3,AL031854,1854,9,7,12,28.0,-78.6,hurricane,3,110,938,NA,NA,Central of Costa Rica
3,4,AL031854,1854,9,8,18,31.6,-81.1,hurricane,3,100,950,NA,NA,Southern of Cuba
4,5,AL031854,1854,9,8,20,31.7,-81.1,hurricane,3,100,950,NA,NA,Near the east coast of Japón


In [7]:
# Obtener por categoria del evento, cuantos eventos (tormenta, huracan o depresion) se han presentado


query = r"""
select
    category as categoria,
    count(*) as tormentas
from
    anomalias.storms
group by
    1
order by
    2;
"""
df = pd.read_sql(text(query),engine)
df.head()


,categoria,tormentas
0,5,134
1,4,684
2,3,855
3,2,1346
4,1,3233


In [9]:
# Obtener la consulta anterior como porcentaje respecto del total de eventos. Obtener la información solo de huracanes.

query = r"""
select
    category as categoria,
    count(*) as tormentas,
    (select count(*) from anomalias.storms where status="hurricane") as total,
    100*count(*)/(select count(*) from anomalias.storms) as pct_total
from
    anomalias.storms
where
    status="hurricane"
group by
    1
order by
    2;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,categoria,tormentas,total,pct_total
0,5,134,6250,0.7534
1,4,684,6250,3.8457
2,3,855,6250,4.8072
3,2,1345,6250,7.5621
4,1,3232,6250,18.1716


##### 6-4 Anomalias con Estadistica

In [11]:
# Para los eventos de tipo huracan, ordenar por la velocidad del viento y obtener los percentiles */

query = r"""
select
    status,
    wind,
    percent_rank() over (partition by status order by wind) as percentil
from
    anomalias.storms
where
    status="hurricane";
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,status,wind,percentil
0,hurricane,65,0.0
1,hurricane,65,0.0
2,hurricane,65,0.0
3,hurricane,65,0.0
4,hurricane,65,0.0
5,hurricane,65,0.0
6,hurricane,65,0.0
7,hurricane,65,0.0
8,hurricane,65,0.0
9,hurricane,65,0.0


In [12]:
df.tail(10)

,status,wind,percentil
6240,hurricane,155,0.995999
6241,hurricane,155,0.995999
6242,hurricane,155,0.995999
6243,hurricane,160,0.999040
6244,hurricane,160,0.999040
6245,hurricane,160,0.999040
6246,hurricane,160,0.999040
6247,hurricane,160,0.999040
6248,hurricane,160,0.999040
6249,hurricane,165,1.000000


In [13]:
# De la tabla anterior redondear el percentil a 4 decimales y hacer un conteo de cuantos huracanes hubo */

query = r"""
select
    status,
    wind,
    round(percentil,4),
    count(*)
from
    (
    select
        status,
        wind,
        percent_rank() over (partition by status order by wind) as percentil
    from
        anomalias.storms
    where
        status="hurricane"
    ) as a
group by
    1,2,3
order by
    2 desc;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,status,wind,"round(percentil,4)",count(*)
0,hurricane,165,1.0000,1
1,hurricane,160,0.9990,6
2,hurricane,155,0.9960,19
3,hurricane,150,0.9920,25
4,hurricane,145,0.9862,36
5,hurricane,140,0.9787,47
6,hurricane,135,0.9712,47
7,hurricane,130,0.9552,100
8,hurricane,125,0.9336,135
9,hurricane,120,0.9075,163


In [14]:
df.tail(10)

,status,wind,"round(percentil,4)",count(*)
11,hurricane,110,0.8304,243
12,hurricane,105,0.7905,249
13,hurricane,100,0.7324,363
14,hurricane,95,0.6809,322
15,hurricane,90,0.5918,557
16,hurricane,85,0.5172,466
17,hurricane,80,0.4215,598
18,hurricane,75,0.3044,732
19,hurricane,70,0.1797,779
20,hurricane,65,0.0000,1123


In [17]:
# Ordenando de manera ascendente por "wind" obtener 100 buckets con aproximadamente el mismo numero de elementos cada uno */

query = r"""
select
    status,
    wind,
    ntile(100) over (partition by status order by wind) as bucket
from
    anomalias.storms
where
    status="hurricane"
order by
    1,2 desc;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,status,wind,bucket
0,hurricane,165,100
1,hurricane,160,100
2,hurricane,160,100
3,hurricane,160,100
4,hurricane,160,100
5,hurricane,160,100
6,hurricane,160,100
7,hurricane,155,100
8,hurricane,155,100
9,hurricane,155,100


In [24]:
df.tail(10)

,status,wind,bucket
6240,hurricane,65,18
6241,hurricane,65,18
6242,hurricane,65,18
6243,hurricane,65,18
6244,hurricane,65,18
6245,hurricane,65,18
6246,hurricane,65,18
6247,hurricane,65,18
6248,hurricane,65,18
6249,hurricane,65,18


In [27]:
# Ordenando de manera ascendente por "wind" obtener 4 buckets con aproximadamente el mismo numero de elementos cada uno */

query = r"""
select
    status,
    wind,
    ntile(4) over (partition by status order by wind) as bucket
from
    anomalias.storms
where
    status="hurricane"
order by
    1,2 desc;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,status,wind,bucket
0,hurricane,165,4
1,hurricane,160,4
2,hurricane,160,4
3,hurricane,160,4
4,hurricane,160,4
5,hurricane,160,4
6,hurricane,160,4
7,hurricane,155,4
8,hurricane,155,4
9,hurricane,155,4


In [28]:
df.tail(10)

,status,wind,bucket
6240,hurricane,65,1
6241,hurricane,65,1
6242,hurricane,65,1
6243,hurricane,65,1
6244,hurricane,65,1
6245,hurricane,65,1
6246,hurricane,65,1
6247,hurricane,65,1
6248,hurricane,65,1
6249,hurricane,65,1


In [30]:
# Con la consulta anterior obtener donde empieza y termina cada bucket */

query = r"""
select
    bucket,
    min(wind) as minimo,
    max(wind) as maximo
from 
    (
    select
        status,
        wind,
        ntile(4) over (partition by status order by wind) as bucket
    from
        anomalias.storms
    where
        status="hurricane"
    order by
        1,2 desc) as a
group by 1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,bucket,minimo,maximo
0,4,100,165
1,3,80,100
2,2,70,80
3,1,65,70


In [33]:
# Calcular la desviación estandar de la columna wind */

query = r"""
select
    std(wind) desviacion_std
from
    anomalias.storms;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,desviacion_std
0,26.966443


In [ ]:
# Obtener el z_score = (Xi - Media) / Desviación Estandar */

query = r"""
select
    wind,
    avg(wind) over () as media,
    round(std(wind) over (),4) as desv,
    round( (wind - avg(wind) over ()) / round(std(wind) over (),4) ,4) as z_score
from
    anomalias.storms
order by
    1 desc;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,wind,media,desv,z_score
0,165,56.3697,26.9664,4.0284
1,160,56.3697,26.9664,3.8429
2,160,56.3697,26.9664,3.8429
3,160,56.3697,26.9664,3.8429
4,160,56.3697,26.9664,3.8429
5,160,56.3697,26.9664,3.8429
6,160,56.3697,26.9664,3.8429
7,155,56.3697,26.9664,3.6575
8,155,56.3697,26.9664,3.6575
9,155,56.3697,26.9664,3.6575
